In [1]:
# Locate the RAG files

from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

config_files = list(INPUT_ROOT.rglob("taxify_config.json"))
chunk_files = list(INPUT_ROOT.rglob("taxify_chunks.jsonl"))
index_files = list(INPUT_ROOT.rglob("taxify_faiss.index"))
manifest_files = list(INPUT_ROOT.rglob("vector_manifest.json"))

print("Configuration:", config_files)
print("Chunks:", chunk_files)
print("FAISS index:", index_files)
print("Vector manifest:", manifest_files)

assert config_files, "Configuration file not found."
assert chunk_files, "Chunk metadata not found."
assert index_files, "FAISS index not found."
assert manifest_files, "Vector manifest not found."

CONFIG_PATH = config_files[0]
CHUNKS_PATH = chunk_files[0]
FAISS_INDEX_PATH = index_files[0]
VECTOR_MANIFEST_PATH = manifest_files[0]

print("\nSelected configuration:", CONFIG_PATH)
print("Selected chunks:", CHUNKS_PATH)
print("Selected FAISS index:", FAISS_INDEX_PATH)
print("Selected manifest:", VECTOR_MANIFEST_PATH)

print("\nNotebook 03 artifacts found successfully.")

Configuration: [PosixPath('/kaggle/input/notebooks/nallagantisharath/vector-rag-database/taxify/taxify_config.json')]
Chunks: [PosixPath('/kaggle/input/notebooks/nallagantisharath/vector-rag-database/taxify/processed/taxify_chunks.jsonl')]
FAISS index: [PosixPath('/kaggle/input/notebooks/nallagantisharath/vector-rag-database/taxify/vectors/taxify_faiss.index')]
Vector manifest: [PosixPath('/kaggle/input/notebooks/nallagantisharath/vector-rag-database/taxify/vectors/vector_manifest.json')]

Selected configuration: /kaggle/input/notebooks/nallagantisharath/vector-rag-database/taxify/taxify_config.json
Selected chunks: /kaggle/input/notebooks/nallagantisharath/vector-rag-database/taxify/processed/taxify_chunks.jsonl
Selected FAISS index: /kaggle/input/notebooks/nallagantisharath/vector-rag-database/taxify/vectors/taxify_faiss.index
Selected manifest: /kaggle/input/notebooks/nallagantisharath/vector-rag-database/taxify/vectors/vector_manifest.json

Notebook 03 artifacts found successfully.

In [2]:
# Install Llama and RAG dependencies

!pip install -q \
    faiss-cpu \
    sentence-transformers \
    transformers \
    accelerate \
    bitsandbytes \
    huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 44.0 MB/s eta 0:00:00


In [3]:
import torch
import transformers
import faiss
import sentence_transformers
import accelerate
import bitsandbytes

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("SentenceTransformers:", sentence_transformers.__version__)
print("FAISS:", faiss.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)

print("\nCUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for gpu_number in range(torch.cuda.device_count()):
    print(
        f"GPU {gpu_number}:",
        torch.cuda.get_device_name(gpu_number)
    )

assert torch.cuda.is_available(), "A GPU is required for Llama inference."

print("\nLlama and RAG environment verification passed.")

PyTorch: 2.10.0+cu128
Transformers: 5.0.0
SentenceTransformers: 5.4.1
FAISS: 1.15.1
Accelerate: 1.13.0
BitsAndBytes: 0.50.2

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

Llama and RAG environment verification passed.


In [4]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

hf_token = UserSecretsClient().get_secret("Taxify-Kaggle")

assert hf_token, "The Taxify-Kaggle secret is missing or disabled."

login(
    token=hf_token,
    add_to_git_credential=False
)

account = whoami(token=hf_token)

print("Hugging Face account:", account["name"])
print("Hugging Face token loaded securely.")

Hugging Face account: NallagantiSharath
Hugging Face token loaded securely.


In [5]:
import json
import faiss
import torch
from sentence_transformers import SentenceTransformer

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    config = json.load(file)

with open(CHUNKS_PATH, "r", encoding="utf-8") as file:
    chunks = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

with open(VECTOR_MANIFEST_PATH, "r", encoding="utf-8") as file:
    vector_manifest = json.load(file)

faiss_index = faiss.read_index(str(FAISS_INDEX_PATH))

embedding_device = "cuda:0" if torch.cuda.is_available() else "cpu"

embedding_model = SentenceTransformer(
    config["embedding_model"],
    device=embedding_device
)

embedding_model.max_seq_length = 256

print("Project:", config["project_name"])
print("Chunks loaded:", len(chunks))
print("Vectors loaded:", faiss_index.ntotal)
print("Vector dimension:", faiss_index.d)
print("Embedding device:", embedding_device)

assert len(chunks) == 3656
assert faiss_index.ntotal == len(chunks)
assert faiss_index.d == 384

print("Taxify RAG database loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Project: Taxify
Chunks loaded: 3656
Vectors loaded: 3656
Vector dimension: 384
Embedding device: cuda:0
Taxify RAG database loaded successfully.


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"
LLM_DEVICE = 1 if torch.cuda.device_count() > 1 else 0

print("Loading model:", MODEL_NAME)
print("LLM GPU:", LLM_DEVICE)
print("This download is approximately 6.5 GB.")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=hf_token
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=hf_token,
    dtype=torch.float16,
    device_map={"": LLM_DEVICE},
    low_cpu_mem_usage=True
)

llm_model.eval()

print("\nModel loaded:", MODEL_NAME)
print("Model device:", next(llm_model.parameters()).device)
print("Model data type:", next(llm_model.parameters()).dtype)
print("Llama 3.2 loaded successfully.")

Loading model: unsloth/Llama-3.2-3B-Instruct
LLM GPU: 1
This download is approximately 6.5 GB.


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]


Model loaded: unsloth/Llama-3.2-3B-Instruct
Model device: cuda:1
Model data type: torch.float16
Llama 3.2 loaded successfully.


In [7]:
# Test standalone Llama generation

test_messages = [
    {
        "role": "system",
        "content": (
            "You are Taxify, a helpful educational assistant. "
            "Answer clearly and concisely."
        )
    },
    {
        "role": "user",
        "content": "What is artificial intelligence?"
    }
]

model_inputs = tokenizer.apply_chat_template(
    test_messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(llm_model.device)

with torch.inference_mode():
    generated_tokens = llm_model.generate(
        **model_inputs,
        max_new_tokens=120,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

new_tokens = generated_tokens[0][
    model_inputs["input_ids"].shape[-1]:
]

test_answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
).strip()

print("Question: What is artificial intelligence?")
print("\nLlama answer:")
print(test_answer)

assert test_answer
print("\nStandalone Llama generation passed.")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Question: What is artificial intelligence?

Llama answer:
Artificial intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as:

1. Learning: AI systems can learn from data and improve their performance over time.
2. Problem-solving: AI systems can analyze problems and find solutions.
3. Reasoning: AI systems can draw conclusions based on data and rules.
4. Perception: AI systems can interpret and understand data from sensors and other sources.

AI systems use algorithms and data to make decisions, classify objects, and generate insights. They can be categorized into two main types:

1. Narrow or

Standalone Llama generation passed.


In [8]:
TEST_QUESTION = "Explain the difference between a computer virus and ransomware."

test_messages = [
    {
        "role": "system",
        "content": (
            "You are Taxify, a helpful educational assistant. "
            "Answer clearly and concisely."
        )
    },
    {
        "role": "user",
        "content": TEST_QUESTION
    }
]

model_inputs = tokenizer.apply_chat_template(
    test_messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(llm_model.device)

with torch.inference_mode():
    generated_tokens = llm_model.generate(
        **model_inputs,
        max_new_tokens=150,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

new_tokens = generated_tokens[0][
    model_inputs["input_ids"].shape[-1]:
]

test_answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
).strip()

print("Question:", TEST_QUESTION)
print("\nLlama answer:")
print(test_answer)

assert test_answer
print("\nStandalone Llama generation passed.")

Question: Explain the difference between a computer virus and ransomware.

Llama answer:
I'd be happy to explain the difference between a computer virus and ransomware.

**Computer Virus:**
A computer virus is a type of malware that replicates itself by attaching to other programs or files on a computer. It can cause various types of damage, such as:

* Crashing the system
* Stealing sensitive information
* Displaying unwanted messages or ads
* Disrupting normal system operations

However, a virus typically doesn't demand payment in exchange for restoring access to data.

**Ransomware:**
Ransomware is a specific type of malware that not only replicates itself but also encrypts a victim's files or locks their device. It demands a ransom, usually in cryptocurrency, to restore access to the data or

Standalone Llama generation passed.
